# Ropedia Academy — D4 · Semantic & open-vocabulary mapping

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/D4.ipynb)

> **Runs Bayesian multi-frame label fusion (with open-vocabulary CLIP matching) and plots the per-class posterior sharpening toward the true class over frames.**
>
> 运行贝叶斯多帧标签融合（含开放词汇 CLIP 匹配），并画出逐帧各类别后验向真实类别收敛。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/D4

In [ ]:
import torch, torch.nn.functional as F, matplotlib.pyplot as plt

# Fuse per-frame 2D labels into a 3D element via BAYESIAN multi-frame fusion.
n_classes, true_class = 4, 2; torch.manual_seed(0)
logp = torch.zeros(n_classes); history = []
for _ in range(15):                              # 15 noisy 2D observations
    obs = F.one_hot(torch.tensor(true_class), n_classes).float()*2 + torch.randn(n_classes)
    logp = logp + F.log_softmax(obs, -1); logp = logp - logp.logsumexp(0)   # renormalize
    history.append(logp.exp().tolist())
print("fused posterior:", [round(p,3) for p in logp.exp().tolist()], "-> argmax", logp.argmax().item())

# Open-vocabulary: match a language query to per-element CLIP features.
voxel_clip = F.normalize(torch.randn(5, 16), dim=1); query = F.normalize(torch.randn(16), dim=0)
print("language-query best match: element", (voxel_clip @ query).argmax().item())

# Visualize the posterior over classes sharpening as evidence accumulates.
import numpy as np; H = np.array(history)
plt.figure(figsize=(6, 3.3))
for c in range(n_classes): plt.plot(H[:, c], label=f"class {c}" + (" (true)" if c==true_class else ""))
plt.title("Bayesian label fusion: multi-frame voting sharpens the belief")
plt.xlabel("frame"); plt.ylabel("P(class)"); plt.legend(fontsize=8); plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/D4
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks